<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b> Chat Memory Patterns</b></font> </br></p>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 130.211.243.23
Hostname: 23.243.211.130.bc.googleusercontent.com
Stadt: Taipei
Region: Taiwan
Land: TW
Koordinaten: 25.0531,121.5264
Provider: AS396982 Google LLC
Postleitzahl: None
Zeitzone: Asia/Taipei


# 1 | Intro
---

<p><font color='black' size="5">
Zustandslosigkeit von LLMs
</font></p>

Large Language Models (LLMs) wie GPT sind von Natur aus **zustandslos** - sie verfugen uber kein eingebautes Gedachtnis. Jede Anfrage wird isoliert verarbeitet, ohne Bezug zu vorherigen Interaktionen. Deshalb muss der Chatverlauf (Historie) bei jeder Anfrage neu ubergeben werden.

```
Ohne Memory:
User: "Mein Name ist Max"
AI: "Hallo Max!"
User: "Wie heisse ich?"
AI: "Das habe ich nicht gespeichert."
```

**Gängige Memory-Patterns:**

| Pattern | Beschreibung | Anwendungsfall |
|---------|--------------|----------------|
| **Python-Liste** | Einfachste Lösung | Prototyping, kurze Sessions |
| **Trimming** | Nur letzte N Nachrichten an LLM | Token-Limit einhalten |
| **Summary** | Alte Nachrichten zusammenfassen | Lange Sessions, Kontext erhalten |
| **SqliteSaver** | Persistente Speicherung | Production, Multi-User |

In [2]:
#@markdown Chat Memory Patterns
diagram = """
%%{init: {'theme':'forest'}}%%
graph TD
    root[Chat Memory Patterns]

    %% 1. Manuelle Variante
    root -->|Short-term / Manuell| manual[Python-Liste]
    manual --> manual_desc["Einfache Liste im RAM<br/>Use Case: Prototyping, kurze Sessions"]

    %% 2. LangGraph Variante
    root -->|Automatisch / LangGraph| lg[StateGraph + MemorySaver]

    %% Optimierungs-Strategien
    lg -->|Context Management| strat[Strategien mit Token-Limits]

    strat --> trim[Trimming]
    trim --> trim_desc["Nur letzte N Nachrichten an LLM<br/>State bleibt vollständig erhalten"]

    strat --> sum[Summary]
    sum --> sum_desc["RemoveMessage löscht alte Nachrichten<br/>Summary als SystemMessage im State"]

    %% Persistenz-Lösungen
    lg -->|Long-term Memory| persist[Persistenz-Backends]

    persist --> sqlite[SqliteSaver]
    sqlite --> sqlite_desc["SQLite-Datenbank<br/>Use Case: Persistenz, Multi-Thread"]

    %% Styling
    style root fill:#2ecc71,stroke:#27ae60,stroke-width:2px,color:white
    style lg fill:#3498db,stroke:#2980b9,stroke-width:2px,color:white
    style manual fill:#95a5a6,stroke:#7f8c8d,stroke-width:2px,color:white
    style strat fill:#f1c40f,stroke:#f39c12,color:black
    style persist fill:#e67e22,stroke:#d35400,color:white
"""
mermaid(diagram, width=800)


# 2 | Short-term Memory (Python-Liste)
---

Die einfachste Form von Memory: Eine **Python-Liste**, die alle Nachrichten speichert und bei jedem API-Call mitgesendet wird.

In [3]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ── Modell ────────────────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE

In [4]:
# System-Prompt
system_prompt = "Du bist ein hilfreicher und humorvoller KI-Assistent."

# Prompt-Template mit Historie (MessagesPlaceholder nimmt die Historie entgegen)
prompt = ChatPromptTemplate.from_messages([
    ("system", "{system_prompt}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])

# Modell
llm = init_chat_model(BASELINE)

# Parser
parser = StrOutputParser()

# Chain
chain = prompt | llm | parser

In [5]:
# Chat-Funktion mit manueller Historien-Verwaltung
def chat(chat_history, user_input):
    """Fuhrt eine Chat-Interaktion mit manueller Historien-Verwaltung durch."""

    # Chain aufrufen (Historie wird im Prompt mitgeschickt)
    response = chain.invoke({
        'system_prompt': system_prompt,
        'chat_history': chat_history,
        'user_input': user_input
    })

    # Ausgabe
    mprint(f"### 🧑‍🦱 Mensch:\n{user_input}")
    mprint(f"### 🤖 KI:\n{response}\n")

    # Memory (Liste) MANUELL aktualisieren
    chat_history.extend([HumanMessage(content=user_input), AIMessage(content=response)])

    return chat_history

In [6]:
# Historie initialisieren
chat_history = [SystemMessage(content=system_prompt)]

# Konversation
chat_history = chat(chat_history, "Mein Name ist Max")
chat_history = chat(chat_history, "Ich mag Python-Programmierung")
chat_history = chat(chat_history, "Weisst du noch, wie ich heisse und was ich mag?")

### 🧑‍🦱 Mensch:
Mein Name ist Max

### 🤖 KI:
Hey Max! 😄  
Willkommen! Wie kann ich dir heute helfen—suchst du Infos, willst du was planen, oder brauchst du einfach nur ein bisschen KI-Input für den Alltag?


### 🧑‍🦱 Mensch:
Ich mag Python-Programmierung

### 🤖 KI:
Nice, Max! Python ist wie ein Schweizer Taschenmesser—nur ohne Berge, aber mit sehr viel „Warum geht das so easy?!“ 😄🐍

Was genau willst du mit Python machen?

- **Anfänger:** erste Schritte, Syntax, Datenstrukturen?
- **Projekte:** z.B. Webscraping, Telegram/Discord-Bots, Web-Apps (Flask/FastAPI), Automatisierung?
- **Daten:** Pandas, Matplotlib, ML?
- **Spielen:** kleine Games, Pygame?
- **Best Practices:** Tests, Virtual Envs, Logging, Code-Struktur?

Sag mir kurz dein **aktuelles Level** (Anfänger / mittel / fortgeschritten) und **welches Projekt** du im Kopf hast—dann schlage ich dir den nächsten sinnvollen Schritt vor.


### 🧑‍🦱 Mensch:
Weisst du noch, wie ich heisse und was ich mag?

### 🤖 KI:
Klar! 😊  
Du heißt **Max** und du magst **Python-Programmierung**.  

Willst du jetzt an einem Projekt basteln oder erst dein Python-Wissen ein bisschen auffrischen? 😄


In [7]:
mprint("### Gespeicherte Nachrichten (Liste):\n---")
for msg in chat_history:
    mprint(f"  **{msg.type}**:   {msg.content}")

### Gespeicherte Nachrichten (Liste):
---

  **system**:   Du bist ein hilfreicher und humorvoller KI-Assistent.

  **human**:   Mein Name ist Max

  **ai**:   Hey Max! 😄  
Willkommen! Wie kann ich dir heute helfen—suchst du Infos, willst du was planen, oder brauchst du einfach nur ein bisschen KI-Input für den Alltag?

  **human**:   Ich mag Python-Programmierung

  **ai**:   Nice, Max! Python ist wie ein Schweizer Taschenmesser—nur ohne Berge, aber mit sehr viel „Warum geht das so easy?!“ 😄🐍

Was genau willst du mit Python machen?

- **Anfänger:** erste Schritte, Syntax, Datenstrukturen?
- **Projekte:** z.B. Webscraping, Telegram/Discord-Bots, Web-Apps (Flask/FastAPI), Automatisierung?
- **Daten:** Pandas, Matplotlib, ML?
- **Spielen:** kleine Games, Pygame?
- **Best Practices:** Tests, Virtual Envs, Logging, Code-Struktur?

Sag mir kurz dein **aktuelles Level** (Anfänger / mittel / fortgeschritten) und **welches Projekt** du im Kopf hast—dann schlage ich dir den nächsten sinnvollen Schritt vor.

  **human**:   Weisst du noch, wie ich heisse und was ich mag?

  **ai**:   Klar! 😊  
Du heißt **Max** und du magst **Python-Programmierung**.  

Willst du jetzt an einem Projekt basteln oder erst dein Python-Wissen ein bisschen auffrischen? 😄

<p><font color='darkblue' size="4">
Problem:
</font></p>

- Keine automatische Session-Verwaltung (Multi-User)
- Manuelles Memory-Management fehleranfallig
- **Bei langen Konversationen: Token-Limit wird uberschritten!**

# 3 | LangGraph MemorySaver
---

Die manuelle Verwaltung von Chat-Historien über Listen oder Dictionaries stößt bei komplexeren Anwendungen schnell an ihre Grenzen. Hier bietet das moderne LangGraph-Framework eine elegante und automatisierte Lösung:

Mit `StateGraph` und `MessagesState` läuft die Chat-Historie komplett automatisch. Nach jedem Schritt speichert ein `Checkpointer` (≘ Kontrollpunkt/Sicherungspunkt) den gesamten Zustand inklusive aller Nachrichten und lädt ihn beim nächsten Aufruf wieder.

Welcher Speicher genutzt wird, entscheidet der Checkpointer: MemorySaver sichert die Daten flüchtig im RAM, während der SqliteSaver sie dauerhaft in einer SQLite-Datei ablegt.

```python
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START
```

`MessagesState` ist ein fertiger Zustandstyp (≘ Speicher-Format) in LangGraph. Er besitzt ein `messages`-Feld, das neue Nachrichten über add_messages vollautomatisch anhängt. Ein manuelles Aktualisieren ist nicht mehr nötig.


```python
from langgraph.graph import MessagesState
```

<p><font color='black' size="5">
3.1 | Basismodell
</font></p>

---

In [8]:
# Import
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START

In [9]:
#@markdown Funktionsaufrufe (LangGraph)
diagram = """
flowchart LR
    %% Definition der Stile für Knoten
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef graphStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    %% 1. Bereich: Benutzereingabe / Schleife (Blau)
    subgraph UI ["<b>1. Input / Output (Code-Funktionen)</b>"]
        direction TB
        Start["`<b>ask(thread_id, user_input)</b><br>Startet Chat via <i>app.invoke()</i>`"]:::loopStyle
        Ende["`<b>mprint() / Ausgabe</b><br>Gibt <i>response</i> im Terminal aus`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    %% 2. Bereich: LangGraph Kernprozess (Orange)
    subgraph Engine ["<b>2. LangGraph-Zustandsmaschine (app)</b>"]
        direction TB
        Load["`<b>1. State laden</b><br>Holt <i>MessagesState</i> für thread_id`"]:::graphStyle
        LLM["`<b>2. Node: 'model' (call_model)</b><br>Verarbeitet <i>SystemMessage + state['messages']</i>`"]:::graphStyle
        Save["`<b>3. State speichern</b><br>Hängt <i>AIMessage</i> via Reducer an`"]:::graphStyle

        Load --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    %% 3. Bereich: Der Speicher (Lila)
    subgraph Storage ["<b>3. Checkpointer (memory)</b>"]
        Memory["`<b>memory = MemorySaver()</b><br>Speichert Zustände im RAM`"]:::saveStyle
    end
    style Storage fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    %% Verbindungen zwischen den Bereichen
    Start --> Load
    Load <--> Memory
    Save --> Memory
    Save --> Ende
"""
mermaid(diagram, width=1100)


<p><font color='violet' size="4">
3. Checkpointer (memory)
</font></p>

In [10]:
# MemorySaver sichert die Daten  im RAM
memory = MemorySaver()

<p><font color='orange' size="4">
2. LangGraph-Zustandsmaschine (app)
</font></p>

In [11]:
# Node-Definition: Bereitet Nachrichten vor und ruft das LLM auf
def call_model(state: MessagesState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    return {"messages": [llm.invoke(messages)]}

In [12]:
# Zustandsgraph definieren und Nodes/Edges verknüpfen
workflow = StateGraph(MessagesState)
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")

In [13]:
# Zusammenfügen der Bausteine der Zustandsmaschine und Kopplung mit dem Checkpointer
app = workflow.compile(checkpointer=memory)
print("StateGraph mit MemorySaver definiert ✅")

StateGraph mit MemorySaver definiert ✅


In [14]:
# Hilfsfunktion: Expliziten Zustand (State) aus dem Checkpointer auslesen
def get_thread_history(thread_id: str) -> list:
    """Gibt die gespeicherten Nachrichten eines Threads zurück."""
    config = {"configurable": {"thread_id": thread_id}}
    state = app.get_state(config)
    return state.values.get("messages", [])

<p><font color='blue' size="4">
1. Input/Output
</font></p>

In [15]:
def ask(thread_id: str, user_input: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    response = result["messages"][-1].content
    mprint(f"**[{thread_id}] Mensch:** {user_input}")
    mprint(f"**[{thread_id}] KI:** {response}\n")
    return response

In [16]:
mprint("## MemorySaver Demo")
mprint("---")

for thread_id, user_input in [
    ("max", "Hallo! Ich bin Max aus München."),
    ("max", "Ich programmiere gerne in Python."),
    ("emma", "Hi! Ich bin Emma und mag Machine Learning."),
    ("max", "Woher komme ich und was ist mein Hobby?"),
]:
    ask(thread_id, user_input)

mprint("### Gespeicherte Threads")
for thread_id in ["max", "emma"]:
    history = get_thread_history(thread_id)
    mprint(f"- **{thread_id}**: {len(history)} Nachrichten")

mprint("### Historie: max")
for index, message in enumerate(get_thread_history("max"), 1):
    role = "Mensch" if message.type == "human" else "KI"
    mprint(f"{index}. **{role}:** {message.content}")

## MemorySaver Demo

---

**[max] Mensch:** Hallo! Ich bin Max aus München.

**[max] KI:** Hey Max! 😄 Schön, dich kennenzulernen. München grüßt München!  

Was steht heute an—brauchst du Hilfe bei irgendwas, oder willst du einfach nur ein bisschen quatschen?


**[max] Mensch:** Ich programmiere gerne in Python.

...

2. **KI:** Hey Max! 😄 Schön, dich kennenzulernen. München grüßt München!  

Was steht heute an—brauchst du Hilfe bei irgendwas, oder willst du einfach nur ein bisschen quatschen?

3. **Mensch:** Ich programmiere gerne in Python.

4. **KI:** Nice, Max! 😄 Python ist quasi die Schweiz unter den Programmiersprachen: sauber, vielseitig und immer einsatzbereit.

Womit hast du gerade zu tun—eher was mit
- Daten/Plots,
- Web (z.B. Flask/FastAPI),
- Automatisierung/Skripte,
- oder etwas komplett anderes?

Wenn du willst, kannst du mir auch kurz sagen, welches Projekt du vorhast (oder den relevanten Code-Schnipsel schicken), dann helfe ich dir gern—inklusive “Debuggen mit Humor”, versprochen.

5. **Mensch:** Woher komme ich und was ist mein Hobby?

6. **KI:** Du kommst aus München (du hast dich so vorgestellt). 🙂

Und dein Hobby ist Programmieren in Python. 😄


<p><font color='black' size="5">
3.2 | Trimming (Sliding Window)
</font></p>

---

**Problem:** Bei langen Konversationen wächst `state["messages"]` unbegrenzt — das Token-Limit des LLM wird überschritten.

**Lösung:** `trim_messages()` im `call_model`-Node begrenzt den an das LLM übergebenen Kontext. Der State bleibt vollständig erhalten — nur das LLM-Kontextfenster wird begrenzt.

> Um auch den State physisch zu kürzen, können `RemoveMessage`-Operationen eingesetzt werden (Abschnitt 3.3).

In [17]:
from langchain_core.messages import trim_messages

MAX_MESSAGES = 6  # maximal 6 Nachrichten im LLM-Kontext (3 Runden)

trimmer = trim_messages(
    max_tokens=MAX_MESSAGES,
    strategy="last",
    token_counter=len,
)

def call_model_trimmed(state: MessagesState):
    """Trimmt den Kontext vor dem LLM-Aufruf — der State bleibt vollständig."""
    trimmed = trimmer.invoke(state["messages"])
    response = llm.invoke([SystemMessage(content=system_prompt)] + trimmed)
    return {"messages": [response]}

workflow_trimmed = StateGraph(MessagesState)
workflow_trimmed.add_node("model", call_model_trimmed)
workflow_trimmed.add_edge(START, "model")

app_trimmed = workflow_trimmed.compile(checkpointer=MemorySaver())
print(f"Graph mit Trimming erstellt (max. {MAX_MESSAGES} Nachrichten im LLM-Kontext) ✅")


Graph mit Trimming erstellt (max. 6 Nachrichten im LLM-Kontext) ✅


In [18]:
mprint("## Trimming Demo (max. 6 Nachrichten im LLM-Kontext)")
mprint("---")

config_trim = {"configurable": {"thread_id": "trim_test"}}

for nr, user_input in enumerate([
    "Mein Name ist Max.",
    "Ich komme aus München.",
    "Ich mag Python.",
    "Ich arbeite als Data Scientist.",
], 1):
    result = app_trimmed.invoke({"messages": [HumanMessage(content=user_input)]}, config=config_trim)
    response = result["messages"][-1].content
    mprint(f"**{nr}. Mensch:** {user_input}")
    mprint(f"**{nr}. KI:** {response}\n")

# State enthält alle Nachrichten — Trimming betrifft nur das LLM-Kontextfenster
state = app_trimmed.get_state(config_trim)
all_messages = state.values.get("messages", [])

mprint("### Vollständige Historie im State")
mprint(f"{len(all_messages)} Nachrichten gespeichert (alle Runden)")
mprint("---")

for index, message in enumerate(all_messages, 1):
    role = "Mensch" if message.type == "human" else "KI"
    mprint(f"{index}. **{role}:** {message.content}")

mprint(f"\n**Ergebnis:** Das Modell sah maximal {MAX_MESSAGES} Nachrichten — der State enthält alle {len(all_messages)}.")


## Trimming Demo (max. 6 Nachrichten im LLM-Kontext)

---

**1. Mensch:** Mein Name ist Max.

**1. KI:** Hey Max! 👋  
Schön dich kennenzulernen. Was steht heute bei dir an—willst du was wissen, planen oder einfach nur ein bisschen quatschen? 😄


**2. Mensch:** Ich komme aus München.

...

5. **Mensch:** Ich mag Python.

6. **KI:** Nice, Max! Python ist wirklich ein bisschen wie eine gute Brezn: einfach, zuverlässig und geht mit allem 😄🥨🐍

Was genau machst du mit Python—eher Daten/ML, Web (z.B. Flask/FastAPI), Automatisierung, oder sowas wie Scraping?

7. **Mensch:** Ich arbeite als Data Scientist.

8. **KI:** Geil! Data Scientist klingt nach „Ich lerne aus Daten und lasse das Chaos ordnen“ — also praktisch Magie mit Mathe-Aufsatz 😄📊

Was nutzt du bei euch am häufigsten: eher **Python + Pandas/NumPy**, **scikit-learn**, **PyTorch/TensorFlow**, **Spark/Databricks** oder sowas wie **Airflow** für Pipelines? Und eher **ML**, **Forecasting**, **NLP** oder **Analytics/BI**?


**Ergebnis:** Das Modell sah maximal 6 Nachrichten — der State enthält alle 8.


<p><font color='black' size="5">
3.3 | Summary (LLM-basierte Zusammenfassung)
</font></p>

---

**Problem:** Trimming verwirft alte Informationen vollständig — wichtiger Kontext geht verloren.

**Alternative:** **Summarization** — Ein LLM fasst alte Nachrichten zusammen, die Zusammenfassung ersetzt sie im State.

**Strategie:**
1. Wenn `state["messages"]` > Limit: Alte Nachrichten zusammenfassen
2. `RemoveMessage` löscht alte Nachrichten physisch aus dem State
3. Summary als `SystemMessage` + letzte N Nachrichten bleiben erhalten

```python
# Pseudo-Code
if len(state["messages"]) > SUMMARY_THRESHOLD:
    summary = llm.invoke("Fasse zusammen: {old_messages}")
    return {
        "messages": [RemoveMessage(id=m.id) for m in old_messages]
                  + [SystemMessage(summary), llm_response]
    }
```

In [19]:
from langchain_core.messages import RemoveMessage

SUMMARY_THRESHOLD = 8
KEEP_RECENT = 4

def summarize_old_messages(old_messages: list) -> str:
    """Fasst alte Nachrichten via LLM zusammen."""
    prompt = "Fasse die Konversation prägnant zusammen. Extrahiere wichtige Fakten (Namen, Orte, Präferenzen):"
    return llm.invoke([SystemMessage(content=prompt)] + old_messages).content

def call_model_summary(state: MessagesState):
    messages = state["messages"]

    if len(messages) > SUMMARY_THRESHOLD:
        to_summarize = messages[:-KEEP_RECENT]
        summary_text = summarize_old_messages(to_summarize)
        summary_msg = SystemMessage(content=f"Zusammenfassung bisher:\n{summary_text}")

        # Alte Nachrichten aus dem State löschen, Summary + neue Antwort einfügen
        remove_ops = [RemoveMessage(id=m.id) for m in to_summarize]
        context = [SystemMessage(content=system_prompt), summary_msg] + list(messages[-KEEP_RECENT:])
        response = llm.invoke(context)
        return {"messages": remove_ops + [summary_msg, response]}

    response = llm.invoke([SystemMessage(content=system_prompt)] + list(messages))
    return {"messages": [response]}

workflow_summary = StateGraph(MessagesState)
workflow_summary.add_node("model", call_model_summary)
workflow_summary.add_edge(START, "model")

app_summary = workflow_summary.compile(checkpointer=MemorySaver())
print(f"Graph mit Summary erstellt (Limit: {SUMMARY_THRESHOLD}, behalte letzte {KEEP_RECENT}) ✅")


Graph mit Summary erstellt (Limit: 8, behalte letzte 4) ✅


In [20]:
mprint("## Summary Demo (Limit: 8 Nachrichten, behalte 4)")
mprint("---")

config_summary = {"configurable": {"thread_id": "summary_test"}}

for i, user_msg in enumerate([
    "Mein Name ist Max.",
    "Ich bin 30 Jahre alt.",
    "Ich komme aus München.",
    "Ich arbeite als Data Scientist.",
    "Mein Hobby ist Fotografie.",
    "Ich mag italienisches Essen.",
], 1):
    result = app_summary.invoke({"messages": [HumanMessage(content=user_msg)]}, config=config_summary)
    mprint(f"**{i}. Mensch:** {user_msg}")
    mprint(f"**{i}. KI:** {result['messages'][-1].content}\n")

# Erinnerungstest
result = app_summary.invoke(
    {"messages": [HumanMessage(content="Fasse zusammen: Wie heisse ich, wie alt bin ich, woher komme ich?")]},
    config=config_summary
)
mprint(f"**Test. Mensch:** Fasse zusammen: Wie heisse ich, wie alt bin ich, woher komme ich?")
mprint(f"**Test. KI:** {result['messages'][-1].content}")

state = app_summary.get_state(config_summary)
msgs_in_state = state.values.get("messages", [])
mprint(f"\n**State nach Summary:** {len(msgs_in_state)} Nachrichten (komprimiert)")
mprint("✅ **Ergebnis:** Der Bot erinnert sich via Summary an alte Infos!")


## Summary Demo (Limit: 8 Nachrichten, behalte 4)

---

**1. Mensch:** Mein Name ist Max.

**1. KI:** Hey Max! 😄 Freut mich, dich kennenzulernen.  
Wie kann ich dir helfen—brauchst du gerade Infos, eine Idee oder einfach nur ein bisschen Chat-Spaß?


**2. Mensch:** Ich bin 30 Jahre alt.

...

**6. KI:** Mensch, das ist ja eine gefährliche Kombi: **Data Science + italienisches Essen + Fotografie** 😄📸🍝  
Da kann nur etwas Gutes rauskommen—entweder im Feed, im Modell oder auf dem Teller.

Welche Art italienisch ist dein Favorit?
- Pizza (welche Richtung: Neapolitan, römisch, dünn/dick?)
- Pasta (Carbonara, Amatriciana, Cacio e pepe…?)
- oder eher Klassiker wie Risotto / Tiramisù?


**Test. Mensch:** Fasse zusammen: Wie heisse ich, wie alt bin ich, woher komme ich?

**Test. KI:** Ich weiß aus dem Verlauf:

- **Wie du heißt:** Max  
- **Wie alt du bist:** Das ist nicht bekannt  
- **Woher du kommst:** Du bist in **München** (herauslesen lässt sich aber nicht, ob du von dort “kommst”).


**State nach Summary:** 6 Nachrichten (komprimiert)

✅ **Ergebnis:** Der Bot erinnert sich via Summary an alte Infos!

# 4 | Long-term Memory
---


<p><font color='black' size="5">
4.1 | SqliteSaver (Persistente Speicherung)
</font></p>

---


**Problem:** `MemorySaver` verliert alle Daten beim Neustart der Laufzeitumgebung (Colab-Session).

**Lösung:** **SqliteSaver** — speichert den Graph-State persistent in einer SQLite-Datenbank. Der Austausch ist minimal:

```python
# Vorher (flüchtig):
app = workflow.compile(checkpointer=MemorySaver())

# Nachher (persistent):
from langgraph.checkpoint.sqlite import SqliteSaver
memory = SqliteSaver.from_conn_string("./sessions.db")
app = workflow.compile(checkpointer=memory)
```

**Vorteile:**
- Daten bleiben nach Neustart erhalten
- Kein eigener Datenbankcode nötig
- Thread-Verwaltung übernimmt der Checkpointer

> **Installation:** `pip install langgraph-checkpoint-sqlite`


In [ ]:
# langgraph-checkpoint-sqlite ist in requirements.txt (genai_lib) enthalten.
# Nur nötig wenn Abschnitt 4 ohne vorherigen Setup-Zellen-Neustart ausgeführt wird:
# !uv pip install --system -q langgraph langgraph-checkpoint langgraph-checkpoint-sqlite --upgrade
from langgraph.checkpoint.sqlite import SqliteSaver
print("SqliteSaver verfügbar ✅")

In [22]:
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH_41 = "./chat_sessions.db"

# SqliteSaver als persistenter Checkpointer — gleicher Graph wie 3.1, nur Checkpointer getauscht
sqlite_memory_41 = SqliteSaver.from_conn_string(DB_PATH_41)
app_persistent = workflow.compile(checkpointer=sqlite_memory_41)

print(f"Graph mit SqliteSaver kompiliert ✅")
print(f"Speicherort: {DB_PATH_41}")


ImportError: cannot import name 'DeltaChannelHistory' from 'langgraph.checkpoint.base' (/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py)

In [ ]:
mprint("## SqliteSaver Demo")
mprint("---")

config_persist = {"configurable": {"thread_id": "max_persistent"}}

for msg in [
    "Hallo! Ich bin Max aus München.",
    "Ich mag Python-Programmierung.",
    "Woher komme ich und was mag ich?",
]:
    result = app_persistent.invoke({"messages": [HumanMessage(content=msg)]}, config=config_persist)
    mprint(f"**Mensch:** {msg}")
    mprint(f"**KI:** {result['messages'][-1].content}\n")


In [ ]:
# Gespeicherten Thread anzeigen
state = app_persistent.get_state(config_persist)
messages = state.values.get("messages", [])

mprint("### Gespeicherter Thread:")
mprint(f"- **max_persistent**: {len(messages)} Nachrichten in `{DB_PATH_41}`")


In [ ]:
mprint("### Test: Persistenz-Check")
mprint("---")

# Neue App-Instanz, gleiche Datenbank (simuliert Neustart)
sqlite_memory_restart = SqliteSaver.from_conn_string(DB_PATH_41)
app_after_restart = workflow.compile(checkpointer=sqlite_memory_restart)

config_restart = {"configurable": {"thread_id": "max_persistent"}}
result = app_after_restart.invoke(
    {"messages": [HumanMessage(content="Was haben wir bisher besprochen?")]},
    config=config_restart
)
mprint(f"**Mensch (nach 'Neustart'):** Was haben wir bisher besprochen?")
mprint(f"**KI:** {result['messages'][-1].content}")
mprint("\n✅ **Ergebnis:** Die Historie überlebt den Neustart!")


<p><font color='black' size="5">
4.2 | SQLite (mehrere Threads)
</font></p>

---

In [ ]:
# Eigene Datenbankklasse entfällt — SqliteSaver übernimmt Persistenz, Threading und Schema
import os
DB_PATH = "./chat_memory.db"


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

persistent_memory = SqliteSaver.from_conn_string(DB_PATH)
app_sqlite = workflow.compile(checkpointer=persistent_memory)
print(f"Persistenter Graph kompiliert ✅  –  Datenbank: {DB_PATH}")


In [ ]:
def chat_sqlite(thread_id: str, user_input: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app_sqlite.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    return result["messages"][-1].content

def show_history(thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    state = app_sqlite.get_state(config)
    messages = state.values.get("messages", [])
    mprint(f"### Thread: {thread_id} ({len(messages)} Nachrichten)")
    mprint("---")
    for i, msg in enumerate(messages, 1):
        role = "Mensch" if msg.type == "human" else "KI"
        mprint(f"{i}. **{role}:** {msg.content}")


In [ ]:
mprint("## SQLite-Chatbot Demo")
mprint("---")

mprint("### Thread: max_session")
mprint(f"**KI:** {chat_sqlite('max_session', 'Hallo! Ich bin Max und komme aus München.')}")
mprint(f"**KI:** {chat_sqlite('max_session', 'Ich interessiere mich für Machine Learning.')}")

mprint("\n### Thread: emma_session")
mprint(f"**KI:** {chat_sqlite('emma_session', 'Hi! Ich bin Emma aus Berlin.')}")

mprint("\n### Zurück zu max_session")
mprint(f"**KI:** {chat_sqlite('max_session', 'Woher komme ich nochmal?')}")


In [ ]:
mprint("### Alle Threads:")
mprint("---")
for thread_id in ["max_session", "emma_session"]:
    config = {"configurable": {"thread_id": thread_id}}
    state = app_sqlite.get_state(config)
    count = len(state.values.get("messages", []))
    mprint(f"- **{thread_id}**: {count} Nachrichten")


In [ ]:
show_history("max_session")


<p><font color='darkblue' size="4">
Test: Neustart-Persistenz
</font></p>

Die Daten bleiben auch nach Neustart erhalten. Führen Sie die nachste Zelle aus, um zu testen:

In [ ]:
# Test: Neue App-Instanz mit gleicher Datenbank (simuliert Neustart)
persistent_memory2 = SqliteSaver.from_conn_string(DB_PATH)
app_sqlite2 = workflow.compile(checkpointer=persistent_memory2)

config_max = {"configurable": {"thread_id": "max_session"}}
result = app_sqlite2.invoke(
    {"messages": [HumanMessage(content="Was war mein Interesse nochmal?")]},
    config=config_max
)
mprint(f"**KI (nach 'Neustart'):** {result['messages'][-1].content}")

# Historie nach Neustart
state = app_sqlite2.get_state(config_max)
msgs = state.values.get("messages", [])
mprint(f"\n### Max's History nach Neustart ({len(msgs)} Nachrichten):")
for i, msg in enumerate(msgs, 1):
    role = "Mensch" if msg.type == "human" else "KI"
    mprint(f"{i}. **{role}:** {msg.content}")


# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**

Teste Trimming-Limits in einem Chatgespräch und beobachte, ab wann der Kontext verloren geht (mind. 5 Nachrichten).

**✅ Erledigt wenn:** Ab einer bestimmten Nachrichtenanzahl gibt das Modell an, die frühere Frage nicht mehr zu kennen — der Grenzwert ist dokumentiert.

In [ ]:
# Grundlagen: Trimming-Limits testen
# Startpunkt: Trimming-Beispiel aus Kapitel 2 als Vorlage

# Ablauf: Chatgespräch mit mindestens 5 Nachrichten führen
# Nach jeder Runde fragen: 'Was war meine erste Frage?'
# Beobachten: Ab wann ist die Antwort 'Das weiß ich nicht mehr'
# Grenzwert dokumentieren
# ...

**Aufbau**

Verbessere den Summary-Prompt oder baue einen interaktiven CLI-Chatbot mit Threads, der Kontext über mehrere Nachrichten hält.

**✅ Erledigt wenn:** Der Chatbot hält Kontext über mindestens fünf aufeinander aufbauende Fragen — eine Rückfrage auf die erste Frage wird korrekt beantwortet.

In [ ]:
# Aufbau: CLI-Chatbot mit Summary oder Threads
# Startpunkt: Memory-Beispiele aus Kapitel 3/4 als Vorlage

# Option A: Summary-Memory — Zusammenfassung statt vollem Verlauf
# Option B: Interaktiver CLI-Chatbot mit while-Schleife und Thread-ID

# Test: 5 aufeinander aufbauende Fragen stellen
# Dann: 'Was war meine erste Frage?' → muss korrekt beantwortet werden
# ...

**Vertiefung**

Kombiniere Trimming, Summary und SQLite zu einem Hybrid-Memory-System für persistente Chatsitzungen.

**✅ Erledigt wenn:** Der Chatbot liest beim Start gespeicherte Sitzungen aus der SQLite-Datenbank — Kontext aus früheren Sitzungen fließt in die Antwort ein.

In [ ]:
# Vertiefung: Hybrid-Memory-System (Trimming + Summary + SQLite)
# Startpunkt: SQLite-Beispiele aus Abschnitt B dieses Notebooks

# 1. Trimming: kurzen Kontext im Arbeitsspeicher halten
# 2. Summary: ältere Nachrichten zusammenfassen und speichern
# 3. SQLite: Sitzungen persistent speichern und beim Start laden

# Test: Colab neu starten → gespeicherter Kontext muss wiederhergestellt werden
# ...

# B | Datenbank auslesen
---

Threads aus `app_sqlite` auslesen — nützlich für Debugging, Analyse oder Export.

In [ ]:
def read_all_threads(thread_ids: list, app_instance=None):
    """Liest alle Threads via LangGraph State API."""
    if app_instance is None:
        app_instance = app_sqlite

    mprint(f"### Alle Threads ({len(thread_ids)} gesamt)")
    mprint("---")

    for thread_id in thread_ids:
        config = {"configurable": {"thread_id": thread_id}}
        state = app_instance.get_state(config)
        messages = state.values.get("messages", [])

        if not messages:
            continue

        mprint(f"#### Thread: {thread_id} ({len(messages)} Nachrichten)")
        for i, msg in enumerate(messages, 1):
            role = "Mensch" if msg.type == "human" else "KI"
            content_short = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
            mprint(f"{i}. **{role}:** {content_short}")
        mprint("")


In [ ]:
read_all_threads(["max_session", "emma_session"])


In [ ]:
import json

def export_thread_to_json(thread_id: str, app_instance=None) -> dict:
    """Exportiert einen Thread als JSON."""
    if app_instance is None:
        app_instance = app_sqlite

    config = {"configurable": {"thread_id": thread_id}}
    state = app_instance.get_state(config)
    messages = state.values.get("messages", [])

    return {
        "thread_id": thread_id,
        "messages": [
            {"role": msg.type, "content": msg.content}
            for msg in messages
        ]
    }

thread_data = export_thread_to_json("max_session")
print(json.dumps(thread_data, indent=2, ensure_ascii=False))


In [ ]:
def delete_db_files():
    """Löscht die SQLite-Datenbanken (Cleanup)."""
    for path in [DB_PATH_41, DB_PATH]:
        if os.path.exists(path):
            os.remove(path)
            print(f"Gelöscht: {path}")
        else:
            print(f"Nicht gefunden: {path}")

# Auskommentiert, um versehentliches Löschen zu verhindern:
# delete_db_files()
